# 🚀 TASK 6.6-R: FINAL LOCKED RECOVERY BENCHMARK TRÊN GOOGLE COLAB PRO
### KIỂM CHUẨN KHÓA SỔ CHẾ ĐỘ KÉP (SINGLE-PASS DUAL-MODE) TRÊN 82.003 ẢNH TEST VÀ 6 BIẾN THỂ DEGRADED

> **Chiến dịch**: Phase 6-R (High-Accuracy Anchored Recovery & Dual Operating Modes)  
> **Người chủ trì**: **Thành viên C (Training & Evaluation Lead)**  
> **Người phối hợp**: **Thành viên B (Architecture Lead)**, **Thành viên A (Data & Infra Lead)**  
> **Môi trường**: Google Colab Pro GPU (NVIDIA T4 / L4 / A100 SXM4) + Google Drive 5TB  
> **Mục tiêu đo đạc**:  
> 1. **Bước 3**: Kiểm chuẩn hậu nghiệm checkpoint WiSE-FT (`fatformer_wise_ft_best.pth`, 4.47 GB, SHA-256 `820585...73f1b`).  
> 2. **Bước 8**: Kiểm chuẩn chính quy checkpoint Anchored Recovery (`fatformer_anchored_recovery_best.pth`, sản sinh từ Task 6.3-R).  
> 3. Tự động đo Single-Pass ở cả 2 chế độ: **Mode 1** ($T^*=0.50$, Standard Balanced) và **Mode 2** ($T^*=0.40$, High-Recall Forensic Q30), đối chiếu trực tiếp với Hard Gates Phase 6-R.

## 🛠️ BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA
> Mount Google Drive, tự động nhận diện thư mục chứa dữ liệu (`FatFormer_Hub` hoặc `Fatformer`), clone/sync kho mã nguồn và cài đặt các thư viện cần thiết.

In [1]:
# ==============================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA
# ==============================================================================
import os, sys, glob, shutil, zipfile, subprocess

print("[*] Đang kết nối Google Drive 5TB...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"[!] Không thể mount Drive tự động: {e}")

# Tự động dò tìm thư mục gốc trên Google Drive
drive_candidates = [
    "/content/drive/MyDrive/FatFormer_Hub",
    "/content/drive/MyDrive/Fatformer",
    "/content/drive/MyDrive/fatformer",
    "/content/drive/MyDrive"
]
DRIVE_ROOT = None
for dc in drive_candidates:
    if os.path.exists(os.path.join(dc, "datasets")):
        DRIVE_ROOT = dc
        break
    if os.path.exists(dc) and any(k in dc for k in ["FatFormer_Hub", "Fatformer"]):
        DRIVE_ROOT = dc
        break

if not DRIVE_ROOT:
    DRIVE_ROOT = drive_candidates[0]
print(f"[✓] Thư mục làm việc trên Drive: {DRIVE_ROOT}")

# Đồng bộ mã nguồn dự án từ GitHub
REPO_DIR = "/content/fatformer-xla"
BRANCH = "hoang"
if not os.path.exists(REPO_DIR):
    print(f"[*] Đang clone kho mã nguồn fatformer-xla từ GitHub (nhánh {BRANCH})...")
    subprocess.run(f"git clone -b {BRANCH} https://github.com/LeeVHoangtk3/fatformer-xla.git {REPO_DIR}", shell=True, check=True)
else:
    print(f"[*] Kho mã nguồn đã tồn tại, đang đồng bộ commit mới nhất từ nhánh {BRANCH}...")
    subprocess.run(f"git -C {REPO_DIR} fetch origin {BRANCH}", shell=True, check=True)
    subprocess.run(f"git -C {REPO_DIR} reset --hard origin/{BRANCH}", shell=True, check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Cài đặt các phụ thuộc cần thiết
%pip install -q ftfy regex tqdm tabulate matplotlib seaborn pandas
print("[✓] Hoàn tất Bước 1: Môi trường Colab sẵn sàng!")


[*] Đang kết nối Google Drive 5TB...
Mounted at /content/drive
[✓] Thư mục làm việc trên Drive: /content/drive/MyDrive/Fatformer
[*] Đang clone kho mã nguồn fatformer-xla từ GitHub (nhánh hoang)...
[✓] Hoàn tất Bước 1: Môi trường Colab sẵn sàng!


## 🔍 BƯỚC 2: KIỂM TRA GPU PHẦN CỨNG & ĐỊNH VỊ CHECKPOINTS, CLIP BACKBONE
> Kiểm tra GPU (yêu cầu T4/L4/A100) và tự động dò tìm:
> - Backbone CLIP `ViT-L-14.pt`
> - Checkpoint WiSE-FT `fatformer_wise_ft_best.pth` (4.47 GB, SHA-256 `820585...73f1b`)
> - Checkpoint Anchored Recovery `fatformer_anchored_recovery_best.pth` (nếu đã huấn luyện xong ở Task 6.3-R)

In [2]:
# ==============================================================================
# BƯỚC 2: KIỂM TRA GPU & ĐỊNH VỊ TÀI NGUÊN TRỌNG SỐ
# ==============================================================================
import torch

assert torch.cuda.is_available(), 'LỖI: Vui lòng bật GPU trong Runtime -> Change runtime type -> T4/L4/A100 GPU!'
gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'[✓] GPU khả dụng: {gpu_name} ({gpu_vram:.2f} GB VRAM) - Sẵn sàng cho Locked Benchmark!')

def find_file(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    return None

# 1. Dò tìm CLIP ViT-L/14 Backbone
clip_path = find_file([
    f'{DRIVE_ROOT}/**/ViT-L-14.pt',
    '/content/**/ViT-L-14.pt',
    '/content/drive/MyDrive/**/ViT-L-14.pt',
    f'{REPO_DIR}/DATASET/pretrained/ViT-L-14.pt'
])
print(f'[1] CLIP ViT-L/14 Backbone: {clip_path}')

# 2. Dò tìm Checkpoint WiSE-FT (Task 6.1-R)
wise_ckpt_path = find_file([
    f'{DRIVE_ROOT}/**/fatformer_wise_ft_best.pth',
    '/content/fatformer_wise_ft_best.pth',
    '/content/drive/MyDrive/**/fatformer_wise_ft_best.pth',
    f'{REPO_DIR}/DATASET/checkpoints/wise_ft/fatformer_wise_ft_best.pth'
])
if wise_ckpt_path:
    size_gb = os.path.getsize(wise_ckpt_path) / (1024**3)
    print(f'[2] Checkpoint WiSE-FT   : {wise_ckpt_path} ({size_gb:.2f} GB) [✓ Sẵn sàng]')
else:
    print('[!] Cảnh báo: Chưa tìm thấy fatformer_wise_ft_best.pth trên Drive! Hãy upload tệp này lên Drive.')

# 3. Dò tìm Checkpoint Anchored Recovery (Task 6.3-R)
anchored_ckpt_path = find_file([
    f'{DRIVE_ROOT}/**/fatformer_anchored_recovery_best.pth',
    '/content/fatformer_anchored_recovery_best.pth',
    '/content/drive/MyDrive/**/fatformer_anchored_recovery_best.pth',
    f'{REPO_DIR}/DATASET/checkpoints/fatformer_anchored_recovery_best.pth'
])
if anchored_ckpt_path:
    size_gb = os.path.getsize(anchored_ckpt_path) / (1024**3)
    print(f'[3] Checkpoint Anchored M6-R: {anchored_ckpt_path} ({size_gb:.2f} GB) [✓ Sẵn sàng]')
else:
    print('[!] Ghi chú: Chưa thấy fatformer_anchored_recovery_best.pth. Tệp này sẽ được huấn luyện ở Task 6.3-R.')


[✓] GPU khả dụng: Tesla T4 (14.56 GB VRAM) - Sẵn sàng cho Locked Benchmark!
[1] CLIP ViT-L/14 Backbone: /content/drive/MyDrive/Fatformer/pretrained/ViT-L-14.pt
[2] Checkpoint WiSE-FT   : /content/drive/MyDrive/Fatformer/checkpoint/fatformer_wise_ft_best.pth (4.37 GB) [✓ Sẵn sàng]
[3] Checkpoint Anchored M6-R: /content/drive/MyDrive/Fatformer/checkpoint/fatformer_anchored_recovery_best.pth (1.84 GB) [✓ Sẵn sàng]


## ⚡ BƯỚC 3: TRÍCH XUẤT DỮ LIỆU Từ GOOGLE DRIVE SANG SSD NVME MÁY ẢO COLAB
> Nhằm loại bỏ hoàn toàn hiện tượng nghẽn cổ chai I/O khi đọc trực tiếp hàng chục nghìn file nhỏ qua Google Drive FUSE mount, toàn bộ các gói `.tar` được giải nén 1 lần duy nhất vào ổ cứng NVMe SSD:
> - **18 tập test Clean**: `/content/dataset_local/test_clean`
> - **6 biến thể Degraded**: `/content/dataset_local/test_degraded`

In [3]:
# ==============================================================================
# BƯỚC 3: GIẢI NÉN DỮ LIỆU TEST BENCHMARK SANG SSD NVME
# ==============================================================================
import tarfile

LOCAL_CLEAN_DIR = "/content/dataset_local/test_clean"
LOCAL_DEG_DIR = "/content/dataset_local/test_degraded"
os.makedirs(LOCAL_CLEAN_DIR, exist_ok=True)
os.makedirs("/content/dataset_local", exist_ok=True)

# 1. Dò tìm các tệp .tar trên Google Drive
tar_gans = find_file([
    f'{DRIVE_ROOT}/datasets/test_benchmark_gans.tar',
    f'{DRIVE_ROOT}/**/test_benchmark_gans.tar',
    '/content/drive/MyDrive/**/test_benchmark_gans.tar'
])
tar_diff = find_file([
    f'{DRIVE_ROOT}/datasets/test_benchmark_diffusion.tar',
    f'{DRIVE_ROOT}/**/test_benchmark_diffusion.tar',
    '/content/drive/MyDrive/**/test_benchmark_diffusion.tar'
])
tar_deg = find_file([
    f'{DRIVE_ROOT}/datasets/test_degraded.tar',
    f'{DRIVE_ROOT}/**/test_degraded.tar',
    '/content/drive/MyDrive/**/test_degraded.tar',
    '/content/test_degraded.tar'
])

print(f"[*] Gói GANs tar     : {tar_gans}")
print(f"[*] Gói Diffusion tar: {tar_diff}")
print(f"[*] Gói Degraded tar : {tar_deg}")

# 2. Hàm giải nén an toàn
def extract_tar_fast(tar_path, target_dir):
    if not tar_path or not os.path.exists(tar_path):
        print(f"[!] Bỏ qua: Không tìm thấy {tar_path}")
        return
    print(f"[*] Đang trích xuất {os.path.basename(tar_path)} sang {target_dir}...")
    with tarfile.open(tar_path, 'r') as tf:
        tf.extractall(target_dir)
    print(f"[✓] Đã giải nén xong: {os.path.basename(tar_path)}")

if tar_gans:
    extract_tar_fast(tar_gans, LOCAL_CLEAN_DIR)
if tar_diff:
    extract_tar_fast(tar_diff, LOCAL_CLEAN_DIR)
if tar_deg:
    extract_tar_fast(tar_deg, "/content/dataset_local")

ALL_TARGET_18 = ["progan", "stylegan", "stylegan2", "biggan", "cyclegan", "stargan", "gaugan", "deepfake", "guided", "ldm_200", "ldm_200_cfg", "ldm_100", "glide_50_27", "glide_100_10", "glide_100_27", "dalle", "pndm", "vqdiffusion"]
found_clean_subsets = []
for root, dirs, _ in os.walk(LOCAL_CLEAN_DIR):
    for d in dirs:
        if d.lower() in ALL_TARGET_18 and d.lower() not in [s.lower() for s in found_clean_subsets]:
            found_clean_subsets.append(d)
clean_subsets = found_clean_subsets if len(found_clean_subsets) > 0 else [f for f in os.listdir(LOCAL_CLEAN_DIR) if os.path.isdir(os.path.join(LOCAL_CLEAN_DIR, f))]
deg_variants = [f for f in os.listdir(LOCAL_DEG_DIR) if os.path.isdir(os.path.join(LOCAL_DEG_DIR, f))] if os.path.exists(LOCAL_DEG_DIR) else []

print(f"\n[✓] Các tập Clean sẵn sàng ({len(clean_subsets)}/18): {clean_subsets}")
if len(clean_subsets) == 18:
    print("    • Xác nhận: Toàn bộ 18 tập Clean (8 GANs + 10 Diffusion) đã nằm trọn vẹn trong test/ và diffusion_test/!")
print(f"[✓] Các biến thể Degraded sẵn sàng ({len(deg_variants)}/6): {deg_variants}")


[*] Gói GANs tar     : /content/drive/MyDrive/Fatformer/datasets/test_benchmark_gans.tar
[*] Gói Diffusion tar: /content/drive/MyDrive/Fatformer/datasets/test_benchmark_diffusion.tar
[*] Gói Degraded tar : /content/drive/MyDrive/Fatformer/datasets/test_degraded.tar
[*] Đang trích xuất test_benchmark_gans.tar sang /content/dataset_local/test_clean...


/tmp/ipykernel_842/2051744988.py:40: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(target_dir)


[✓] Đã giải nén xong: test_benchmark_gans.tar
[*] Đang trích xuất test_benchmark_diffusion.tar sang /content/dataset_local/test_clean...
[✓] Đã giải nén xong: test_benchmark_diffusion.tar
[*] Đang trích xuất test_degraded.tar sang /content/dataset_local...
[✓] Đã giải nén xong: test_degraded.tar

[✓] Các tập Clean sẵn sàng (18/18): ['glide_100_27', 'vqdiffusion', 'ldm_200', 'ldm_200_cfg', 'pndm', 'glide_50_27', 'dalle', 'glide_100_10', 'ldm_100', 'guided', 'progan', 'stargan', 'biggan', 'gaugan', 'stylegan', 'deepfake', 'stylegan2', 'cyclegan']
    • Xác nhận: Toàn bộ 18 tập Clean (8 GANs + 10 Diffusion) đã nằm trọn vẹn trong test/ và diffusion_test/!
[✓] Các biến thể Degraded sẵn sàng (6/6): ['down_up', 'jpeg_q50', 'blur_s1', 'jpeg_q30', 'blur_s2', 'jpeg_q70']


## 🎯 BƯỚC 4: THỰC THI KIỂM CHUẨN HẬU NGHIỆM CHECKPOINT WiSE-FT (BƯỚC 3 OPTION B)
> - Nạp checkpoint: `fatformer_wise_ft_best.pth`
> - Chế độ đo: **Single-Pass Dual-Mode** (Mode 1: $T^*=0.50$ & Mode 2: $T^*=0.40$), $T=1.0811$
> - Xuất kết quả có versioning và prefix: `wise_<timestamp>` bảo vệ toàn vẹn báo cáo lịch sử.

In [4]:
# ==============================================================================
# BƯỚC 4: CHẠY KIỂM CHUẨN KHÓA SỔ CHO WiSE-FT CHECKPOINT
# ==============================================================================
import subprocess

OUTPUT_ROOT = "/content/output_task_6.6_r"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Tự động đồng bộ commit mới nhất từ nhánh hoang
subprocess.run(f"git -C {REPO_DIR} fetch origin hoang", shell=True)
subprocess.run(f"git -C {REPO_DIR} checkout -B hoang origin/hoang", shell=True)
subprocess.run(f"git -C {REPO_DIR} reset --hard origin/hoang", shell=True)
eval_script = os.path.join(REPO_DIR, "tools", "eval_locked_benchmark.py")
assert os.path.exists(eval_script), f"LỖI: Không tìm thấy {eval_script}!"

if not wise_ckpt_path or not os.path.exists(wise_ckpt_path):
    raise FileNotFoundError("Không tìm thấy fatformer_wise_ft_best.pth! Vui lòng kiểm tra lại Bước 2.")

cmd_wise = [
    sys.executable, "-u", f"{REPO_DIR}/tools/eval_locked_benchmark.py",
    "--checkpoint", wise_ckpt_path,
    "--clip_path", clip_path or "ViT-L-14.pt",
    "--temperature", "1.0811",
    "--threshold_mode1", "0.50",
    "--threshold_mode2", "0.40",
    "--dual_mode",
    "--output_dir", OUTPUT_ROOT,
    "--versioned",
    "--run_prefix", "wise",
    "--batch_size", "32",
    "--num_workers", "2",
    "--n_bootstrap", "1000",
    "--device", "cuda"
]

has_clean = os.path.exists(LOCAL_CLEAN_DIR) and len(os.listdir(LOCAL_CLEAN_DIR)) > 0
has_deg = os.path.exists(LOCAL_DEG_DIR) and len(os.listdir(LOCAL_DEG_DIR)) > 0

if has_clean:
    cmd_wise.extend(["--test_root", LOCAL_CLEAN_DIR])
if has_deg:
    cmd_wise.extend(["--degraded_root", LOCAL_DEG_DIR, "--samples_per_degraded_subset", "500"])

print("=" * 80)
print(f"[*] Lệnh thực thi WiSE-FT: {' '.join(cmd_wise)}")
print("=" * 80)

process = subprocess.Popen(cmd_wise, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"Kiểm chuẩn WiSE-FT kết thúc với mã lỗi: {process.returncode}")
else:
    print("\n[✓] HOÀN TẤT THỰC THI KIỂM CHUẨN KHÓA SỔ CHO WiSE-FT!")


[*] Lệnh thực thi WiSE-FT: /usr/bin/python3 -u /content/fatformer-xla/tools/eval_locked_benchmark.py --checkpoint /content/drive/MyDrive/Fatformer/checkpoint/fatformer_wise_ft_best.pth --clip_path /content/drive/MyDrive/Fatformer/pretrained/ViT-L-14.pt --temperature 1.0811 --threshold_mode1 0.50 --threshold_mode2 0.40 --dual_mode --output_dir /content/output_task_6.6_r --versioned --run_prefix wise --batch_size 32 --num_workers 2 --n_bootstrap 1000 --device cuda --test_root /content/dataset_local/test_clean --degraded_root /content/dataset_local/test_degraded --samples_per_degraded_subset 500
[*] Kich hoat Versioned Output: /content/output_task_6.6_r/task_6.6_r/wise_20261023_191901
FATFORMER-XLA: FINAL LOCKED BENCHMARK & SINGLE-PASS DUAL-MODE VERIFICATION (TASK 6.6-R)
Tham so hieu chuan dong bang: Temperature T = 1.0811
  -> Mode 1 (Standard Balanced) : Threshold T* = 0.5000
  -> Mode 2 (High-Recall Q30)   : Threshold T* = 0.4000
Thiet bi: cuda | Che do gia lap (Dummy): False
[*] Dang 

## 🏆 BƯỚC 5: THỰC THI KIỂM CHUẨN CHECKPOINT ANCHORED RECOVERY M6-R (BƯỚC 8 OPTION B)
> - Nạp checkpoint: `fatformer_anchored_recovery_best.pth` (sản sinh từ Task 6.3-R Anchored Retraining)
> - Chế độ đo: **Single-Pass Dual-Mode** (Mode 1: $T^*=0.50$ & Mode 2: $T^*=0.40$), $T=1.2763 (Từ Task 6.4-R)$
> - Xuất kết quả có versioning và prefix: `anchored_<timestamp>`.

In [5]:
# ==============================================================================
# BƯỚC 5: CHẠY KIỂM CHUẨN KHÓA SỔ CHO ANCHORED RECOVERY M6-R
# ==============================================================================
# Tự động đồng bộ commit mới nhất từ nhánh hoang
subprocess.run(f"git -C {REPO_DIR} fetch origin hoang", shell=True)
subprocess.run(f"git -C {REPO_DIR} checkout -B hoang origin/hoang", shell=True)
subprocess.run(f"git -C {REPO_DIR} reset --hard origin/hoang", shell=True)
eval_script = os.path.join(REPO_DIR, "tools", "eval_locked_benchmark.py")
assert os.path.exists(eval_script), f"LỖI: Không tìm thấy {eval_script}!"

if not anchored_ckpt_path or not os.path.exists(anchored_ckpt_path):
    print("[!] THÔNG BÁO: Chưa có fatformer_anchored_recovery_best.pth trên ổ đĩa.")
    print("    Sau khi bạn chạy xong Notebook Task 6.3-R để huấn luyện checkpoint này, hãy quay lại chạy ô này!")
else:
    cmd_anchored = [
        sys.executable, "-u", f"{REPO_DIR}/tools/eval_locked_benchmark.py",
        "--checkpoint", anchored_ckpt_path,
        "--clip_path", clip_path or "ViT-L-14.pt",
        # Nhiệt độ tối ưu T = 1.2763 được hiệu chuẩn và nghiệm thu từ Task 6.4-R
        "--temperature", "1.2763",
        "--threshold_mode1", "0.50",
        "--threshold_mode2", "0.40",
        "--dual_mode",
        "--output_dir", OUTPUT_ROOT,
        "--versioned",
        "--run_prefix", "anchored",
        "--batch_size", "32",
        "--num_workers", "2",
        "--n_bootstrap", "1000",
        "--device", "cuda"
    ]
    if has_clean:
        cmd_anchored.extend(["--test_root", LOCAL_CLEAN_DIR])
    if has_deg:
        cmd_anchored.extend(["--degraded_root", LOCAL_DEG_DIR, "--samples_per_degraded_subset", "500"])

    print("=" * 80)
    print(f"[*] Lệnh thực thi Anchored M6-R: {' '.join(cmd_anchored)}")
    print("=" * 80)

    proc_a = subprocess.Popen(cmd_anchored, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc_a.stdout:
        print(line, end="")
    proc_a.wait()
    if proc_a.returncode != 0:
        raise RuntimeError(f"Kiểm chuẩn Anchored Recovery kết thúc với mã lỗi: {proc_a.returncode}")
    else:
        print("\n[✓] HOÀN TẤT THỰC THI KIỂM CHUẨN KHÓA SỔ CHO ANCHORED RECOVERY M6-R!")


[*] Lệnh thực thi Anchored M6-R: /usr/bin/python3 -u /content/fatformer-xla/tools/eval_locked_benchmark.py --checkpoint /content/drive/MyDrive/Fatformer/checkpoint/fatformer_anchored_recovery_best.pth --clip_path /content/drive/MyDrive/Fatformer/pretrained/ViT-L-14.pt --temperature 1.2763 --threshold_mode1 0.50 --threshold_mode2 0.40 --dual_mode --output_dir /content/output_task_6.6_r --versioned --run_prefix anchored --batch_size 32 --num_workers 2 --n_bootstrap 1000 --device cuda --test_root /content/dataset_local/test_clean --degraded_root /content/dataset_local/test_degraded --samples_per_degraded_subset 500
[*] Kich hoat Versioned Output: /content/output_task_6.6_r/task_6.6_r/anchored_20261023_201245
FATFORMER-XLA: FINAL LOCKED BENCHMARK & SINGLE-PASS DUAL-MODE VERIFICATION (TASK 6.6-R)
Tham so hieu chuan dong bang: Temperature T = 1.2763
  -> Mode 1 (Standard Balanced) : Threshold T* = 0.5000
  -> Mode 2 (High-Recall Q30)   : Threshold T* = 0.4000
Thiet bi: cuda | Che do gia lap 

## 📊 BƯỚC 6: TỔNG HỢP KẾT QUẢ, ĐỐI CHIẾU HARD GATES VÀ MA TRẬN SỤT GIẢM
> Đọc các tệp báo cáo JSON thực tế từ `OUTPUT_ROOT/task_6.6_r/` và đối chiếu trực tiếp với khung Hard Gate nghiệm thu Phase 6-R.

In [6]:
# ==============================================================================
# BƯỚC 6: ĐỐI CHIẾU SỐ LIỆU THỰC TẾ VỚI HARD GATES PHASE 6-R
# ==============================================================================
import json
import pandas as pd
from IPython.display import display, Markdown

reports_found = sorted(glob.glob(f"{OUTPUT_ROOT}/task_6.6_r/**/final_locked_benchmark_report.json", recursive=True))
print(f"[*] Tìm thấy {len(reports_found)} báo cáo kiểm chuẩn thực nghiệm:")
for r in reports_found:
    print(f"  -> {r}")

def inspect_report(rp_path):
    with open(rp_path, 'r', encoding='utf-8') as f:
        d = json.load(f)
    meta = d.get('metadata', {})
    m1 = d.get('mode1_standard_balanced', {}).get('summary', {})
    m2 = d.get('mode2_high_recall_forensic', {}).get('summary', {})
    deg_m1 = d.get('mode1_standard_balanced', {}).get('degraded_summary_by_variant', {})
    deg_m2 = d.get('mode2_high_recall_forensic', {}).get('degraded_summary_by_variant', {})

    clean_m1 = m1.get('mean_overall', {})
    clean_m2 = m2.get('mean_overall', {})

    q30_m1 = deg_m1.get('jpeg_q30', {})
    q30_m2 = deg_m2.get('jpeg_q30', {})

    return {
        'run': os.path.basename(os.path.dirname(rp_path)),
        'ckpt': meta.get('checkpoint', 'N/A'),
        'dummy': meta.get('is_dummy_mode', False),
        'm1_macro_acc': clean_m1.get('macro_acc', 0.0),
        'm1_real_acc': clean_m1.get('real_acc', 0.0),
        'm1_fake_acc': clean_m1.get('fake_acc', 0.0),
        'm1_auc': clean_m1.get('auc', 0.0),
        'm1_q30_overall': q30_m1.get('overall_acc', 0.0),
        'm1_q30_real': q30_m1.get('real_acc', 0.0),
        'm2_macro_acc': clean_m2.get('macro_acc', 0.0),
        'm2_fake_acc': clean_m2.get('fake_acc', 0.0),
        'm2_q30_fake': q30_m2.get('fake_acc', 0.0),
    }

if reports_found:
    summary_rows = [inspect_report(rp) for rp in reports_found]
    df_summary = pd.DataFrame(summary_rows)
    display(Markdown("### 📊 Bảng Tổng Hợp Số Liệu Kiểm Chuẩn GPU Thực Tế:"))
    display(df_summary)

    # Kiểm tra Hard Gates trên từng báo cáo
    print("\n" + "=" * 80)
    print("ĐỐI CHIẾU CÁC CHỈ TIÊU HARD GATE PHASE 6-R:")
    print("=" * 80)
    for row in summary_rows:
        print(f"\n[*] Kết quả chạy: {row['run']} (Checkpoint: {row['ckpt']})")
        if row['dummy']:
            print("  [!] CẢNH BÁO: Báo cáo này là DUMMY MODE, không đủ điều kiện nghiệm thu!")
            continue
        g1 = row['m1_macro_acc'] >= 90.0
        g2 = row['m1_real_acc'] >= 90.0
        g3 = row['m1_fake_acc'] >= 88.0
        g4 = row['m1_auc'] >= 95.0
        g5_m1 = row['m1_q30_overall'] >= 60.0 and row['m1_q30_real'] >= 75.0
        g5_m2 = row['m2_q30_fake'] >= 35.0

        print(f"  1. Clean Macro ACC >= 90.0%: {row['m1_macro_acc']:.2f}% -> {'✅ PASS' if g1 else '❌ FAIL'}")
        print(f"  2. Clean Real ACC >= 90.0% : {row['m1_real_acc']:.2f}% -> {'✅ PASS' if g2 else '❌ FAIL'}")
        print(f"  3. Clean Fake ACC >= 88.0% : {row['m1_fake_acc']:.2f}% -> {'✅ PASS' if g3 else '❌ FAIL'}")
        print(f"  4. Clean ROC-AUC >= 95.0%  : {row['m1_auc']:.2f}% -> {'✅ PASS' if g4 else '❌ FAIL'}")
        print(f"  5. Q30 Mode 1 (Ov>=60, R>=75): Ov={row['m1_q30_overall']:.2f}%, R={row['m1_q30_real']:.2f}% -> {'✅ PASS' if g5_m1 else '❌ FAIL'}")
        print(f"  6. Q30 Mode 2 (Fake >= 35%)  : Fake={row['m2_q30_fake']:.2f}% -> {'✅ PASS' if g5_m2 else '❌ FAIL'}")

        all_pass = g1 and g2 and g3 and g4 and g5_m1 and g5_m2
        status_txt = "🎉 ĐẠT TOÀN DIỆN HARD GATE PHASE 6-R (READY TO PASS CHECKLIST)" if all_pass else "⏳ CHƯA ĐẠT TOÀN BỘ CÁC HARD GATE"
        print(f"  ==> KẾT LUẬN: {status_txt}")


[*] Tìm thấy 2 báo cáo kiểm chuẩn thực nghiệm:
  -> /content/output_task_6.6_r/task_6.6_r/anchored_20261023_201245/final_locked_benchmark_report.json
  -> /content/output_task_6.6_r/task_6.6_r/wise_20261023_191901/final_locked_benchmark_report.json


### 📊 Bảng Tổng Hợp Số Liệu Kiểm Chuẩn GPU Thực Tế:

,run,ckpt,dummy,m1_macro_acc,m1_real_acc,m1_fake_acc,m1_auc,m1_q30_overall,m1_q30_real,m2_macro_acc,m2_fake_acc,m2_q30_fake
0,anchored_20261023_201245,fatformer_anchored_recovery_best.pth,False,77.307985,57.006231,97.609739,96.712789,49.09,65.6,71.607600,97.968389,40.42
1,wise_20261023_191901,fatformer_wise_ft_best.pth,False,96.582002,98.097605,95.066399,99.152458,53.37,100.0,96.643298,95.286560,7.36



ĐỐI CHIẾU CÁC CHỈ TIÊU HARD GATE PHASE 6-R:

[*] Kết quả chạy: anchored_20261023_201245 (Checkpoint: fatformer_anchored_recovery_best.pth)
  1. Clean Macro ACC >= 90.0%: 77.31% -> ❌ FAIL
  2. Clean Real ACC >= 90.0% : 57.01% -> ❌ FAIL
  3. Clean Fake ACC >= 88.0% : 97.61% -> ✅ PASS
  4. Clean ROC-AUC >= 95.0%  : 96.71% -> ✅ PASS
  5. Q30 Mode 1 (Ov>=60, R>=75): Ov=49.09%, R=65.60% -> ❌ FAIL
  6. Q30 Mode 2 (Fake >= 35%)  : Fake=40.42% -> ✅ PASS
  ==> KẾT LUẬN: ⏳ CHƯA ĐẠT TOÀN BỘ CÁC HARD GATE

[*] Kết quả chạy: wise_20261023_191901 (Checkpoint: fatformer_wise_ft_best.pth)
  1. Clean Macro ACC >= 90.0%: 96.58% -> ✅ PASS
  2. Clean Real ACC >= 90.0% : 98.10% -> ✅ PASS
  3. Clean Fake ACC >= 88.0% : 95.07% -> ✅ PASS
  4. Clean ROC-AUC >= 95.0%  : 99.15% -> ✅ PASS
  5. Q30 Mode 1 (Ov>=60, R>=75): Ov=53.37%, R=100.00% -> ❌ FAIL
  6. Q30 Mode 2 (Fake >= 35%)  : Fake=7.36% -> ❌ FAIL
  ==> KẾT LUẬN: ⏳ CHƯA ĐẠT TOÀN BỘ CÁC HARD GATE


## 📦 BƯỚC 7: ĐÓNG GÓI KẾT QUẢ VÀ ĐỒNG BỘ VỀ GOOGLE DRIVE 5TB
> Tự động zip toàn bộ thư mục kết quả `OUTPUT_ROOT` và sao chép trực tiếp vào Google Drive: `FatFormer_Hub/logs/task_6.6_r/`.

In [7]:
# ==============================================================================
# BƯỚC 7: ĐỒNG BỘ KẾT QUẢ SANG GOOGLE DRIVE 5TB
# ==============================================================================
export_zip = "/content/task_6.6_r_final_results.zip"
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_ROOT):
        for fn in files:
            fp = os.path.join(root, fn)
            arcname = os.path.relpath(fp, OUTPUT_ROOT)
            zf.write(fp, arcname)

print(f"[✓] Đã nén kết quả thành công: {export_zip} ({os.path.getsize(export_zip)/(1024**2):.2f} MB)")

drive_log_dir = os.path.join(DRIVE_ROOT, "logs", "task_6.6_r")
os.makedirs(drive_log_dir, exist_ok=True)

for item in os.listdir(OUTPUT_ROOT):
    s_item = os.path.join(OUTPUT_ROOT, item)
    d_item = os.path.join(drive_log_dir, item)
    if os.path.isdir(s_item):
        shutil.copytree(s_item, d_item, dirs_exist_ok=True)
    else:
        shutil.copy2(s_item, d_item)

shutil.copy2(export_zip, os.path.join(drive_log_dir, "task_6.6_r_final_results.zip"))
print(f"[✓] ĐÃ ĐỒNG BỘ TOÀN BỘ KẾT QUẢ VÀO GOOGLE DRIVE: {drive_log_dir}")

try:
    from google.colab import files
    files.download(export_zip)
    print("[✓] Đã kích hoạt tải file về máy cá nhân!")
except Exception:
    pass

print("\n🎉 HOÀN TẤT TOÀN DIỆN VÒNG THỰC NGHIỆM TASK 6.6-R TRÊN GOOGLE COLAB PRO!")


[✓] Đã nén kết quả thành công: /content/task_6.6_r_final_results.zip (0.04 MB)
[✓] ĐÃ ĐỒNG BỘ TOÀN BỘ KẾT QUẢ VÀO GOOGLE DRIVE: /content/drive/MyDrive/Fatformer/logs/task_6.6_r


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[✓] Đã kích hoạt tải file về máy cá nhân!

🎉 HOÀN TẤT TOÀN DIỆN VÒNG THỰC NGHIỆM TASK 6.6-R TRÊN GOOGLE COLAB PRO!
